# Tutorial 8 (T8): Naïve Bayes Classification
**Goal:** Build and compare Gaussian, Multinomial, and Bernoulli Naïve Bayes on crop classification.

### Step 1: Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Binarizer
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

sns.set_theme(style="whitegrid")
print("Libraries loaded.")

### Step 2: Load and Split Data

In [ ]:
os.makedirs("data", exist_ok=True)
csv_path = os.path.join("data", "Crop_recommendation.csv")
if not os.path.exists(csv_path):
    crops = ["rice", "maize", "chickpea", "kidneybeans", "pigeonpeas", "mothbeans", "mungbean", "blackgram", "lentil", "pomegranate"]
    df = pd.DataFrame({
        'N': np.random.randint(10, 140, 200),
        'P': np.random.randint(5, 145, 200),
        'K': np.random.randint(15, 205, 200),
        'temperature': np.random.uniform(8.0, 43.0, 200),
        'humidity': np.random.uniform(14.0, 100.0, 200),
        'ph': np.random.uniform(3.5, 9.9, 200),
        'rainfall': np.random.uniform(20.0, 300.0, 200),
        'label': np.random.choice(crops, 200)
    })
    df.to_csv(csv_path, index=False)
else:
    df = pd.read_csv(csv_path)

X = df[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']]
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### Step 3: Train 3 Naïve Bayes Models

In [ ]:
# 1. Gaussian NB (Continuous normal features)
scaler_std = StandardScaler()
X_tr_std = scaler_std.fit_transform(X_train)
X_te_std = scaler_std.transform(X_test)
gnb = GaussianNB()
gnb.fit(X_tr_std, y_train)
y_pred_gnb = gnb.predict(X_te_std)

# 2. Multinomial NB (Non-negative counts/features)
scaler_mm = MinMaxScaler()
X_tr_mm = scaler_mm.fit_transform(X_train)
X_te_mm = scaler_mm.transform(X_test)
mnb = MultinomialNB(alpha=1.0)
mnb.fit(X_tr_mm, y_train)
y_pred_mnb = mnb.predict(X_te_mm)

# 3. Bernoulli NB (Binary features)
binarizer = Binarizer(threshold=0.5)
X_tr_bin = binarizer.fit_transform(X_tr_mm)
X_te_bin = binarizer.transform(X_te_mm)
bnb = BernoulliNB(alpha=1.0)
bnb.fit(X_tr_bin, y_train)
y_pred_bnb = bnb.predict(X_te_bin)

### Step 4: Comparison & Confusion Matrix

In [ ]:
res_df = pd.DataFrame({
    'Model': ['Gaussian NB', 'Multinomial NB', 'Bernoulli NB'],
    'Accuracy (%)': [accuracy_score(y_test, y_pred_gnb)*100, accuracy_score(y_test, y_pred_mnb)*100, accuracy_score(y_test, y_pred_bnb)*100],
    'F1-Score (%)': [f1_score(y_test, y_pred_gnb, average='weighted', zero_division=0)*100,
                     f1_score(y_test, y_pred_mnb, average='weighted', zero_division=0)*100,
                     f1_score(y_test, y_pred_bnb, average='weighted', zero_division=0)*100]
})
display(res_df)

top_classes = sorted(list(set(y_test)))[:8]
cm = confusion_matrix(y_test, y_pred_gnb, labels=top_classes)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=top_classes, yticklabels=top_classes)
plt.title('T8: Confusion Matrix (Gaussian NB)')
plt.show()